# 有量訊號 + 離場追蹤

**用法**：Main 跑完 `_DailySignal` 後執行（Main 最後 `%run _VolumeSignal.ipynb`），或單獨 Restart & Run All。

| 區塊 | 內容 |
|---|---|
| 1. 設定 | 路徑、有量條件、離場參數 |
| 2. 載入 | 讀 `Result/DailyFiles`（追蹤價格）、`Result/checkall`（有量訊號）、有量紀錄、處理進度 |
| 3. 持倉更新 | 補進場價（訊號隔日開盤）、檢查離場、記錄觀察期報酬 |
| 4. 新訊號 | 當日符合「有量」的股票全部收進紀錄 |
| 5. 存檔 | `Result/volume/有量紀錄.csv`、`state.json` |
| 5b. 網頁資料 | `Result/volume/web/` 依訊號月份拆 JSON，給 `volume.html` 讀 |

### 有量條件（同 index.html「有量」頁籤）
成交金額 > 5日均 × 1.3、收漲、LevelArea > -50、ma_up_flag ≥ 1（均線轉強）。
score ≥ 50 不在這裡篩，全部記錄，網頁可勾選比較。

### 離場規則（收盤判斷，依序檢查，任一成立即離場）
1. **停損**：收盤 < 停損價。停損價 = 有量K的最低價，且最多虧 8%（量K低守不住＝那根量是出貨量）
2. **爆量黑K**：成交金額 ≥ 5日均 × 2、收黑且跌 ≥ 3%（主力出貨訊號）
3. **跌破 MA10**：量能行情節奏快，用 MA10 而不是 MA20
4. **移動停利**：曾獲利 ≥ 10% 後，自最高收盤回落 7%
5. **時間出場**：持有滿 20 個交易日
6. **滯漲出場**：持有滿 10 日報酬仍 < 5%（量出來卻推不動，動能失敗）

### 觀察期（用來調整離場點）
不論是否已離場，進場後 20 個交易日內都繼續記錄：**5/10/20 日報酬**、**期間最高%**（用最高價）、**期間最低%**（用最低價）。
累積一段時間後，比對「實際離場報酬」與「抱滿 N 日報酬」、期間最高/最低分布，就能判斷停損/停利該放寬還是收緊。

**處理進度**記在 `state.json`：只處理比上次更新的日期。第一次執行會用現有 DailyFiles 全部回補。

In [ ]:
# =============================================================
# 1. 設定
# =============================================================
import os, glob, json
import numpy as np
import pandas as pd
from datetime import datetime

DAILY_DIR   = 'Result/DailyFiles'   # 每日價格（追蹤用）
CHECK_DIR   = 'Result/checkall'     # 有量訊號來源（Untitled 產出）
OUT_DIR     = 'Result/volume'
LEDGER_PATH = f'{OUT_DIR}/有量紀錄.csv'
STATE_PATH  = f'{OUT_DIR}/state.json'
BASE_INFO   = 'BaseInfo/StockBaseData.csv'

# 有量條件（同 index.html「有量」）
VOL_RATIO = 1.3       # 成交金額 > 5日均 × 1.3
MIN_LEVEL = -50       # LevelArea > -50

# 離場參數
MAX_LOSS      = 0.08     # 停損：量K低，且最多 -8%
DUMP_VOL      = 2.0      # 爆量黑K：成交金額 ≥ 5日均 × 2
DUMP_DROP     = 0.03     #          且收黑、跌幅 ≥ 3%
EXIT_MA       = 'MA_10'  # 跌破均線
TRAIL_TRIGGER = 0.10     # 曾獲利 10% 後啟動移動停利
TRAIL_DD      = 0.07     # 自最高收盤回落 7%
MAX_HOLD_DAYS = 20       # 持有滿 20 個交易日出場
STALL_DAYS    = 10       # 滯漲：持有滿 10 日
STALL_RET     = 0.05     #       報酬仍 < 5%

OBS_DAYS = (5, 10, 20)   # 觀察期：進場後第 N 日報酬（不論是否離場）
OBS_MAX  = max(OBS_DAYS)

LEDGER_COLS = ['訊號日', '代號', '名稱', '市場', '族群', 'score', '量比', '漲幅%', 'ma_up_flag',
               '訊號日收盤', '量K低', '進場日', '進場價', '停損價', '最高收盤', '持有天數',
               '最新日', '最新收盤', '報酬%', '狀態', '離場日', '離場價', '離場原因',
               '追蹤天數', '期間最高%', '期間最低%'] + [f'{n}日%' for n in OBS_DAYS]
os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
# =============================================================
# 2. 載入
# =============================================================
def load_day(path):
    df = pd.read_excel(path, dtype={'stock_number': str})
    df['stock_number'] = df['stock_number'].str.strip()
    return df.drop_duplicates('stock_number', keep='last').set_index('stock_number')


def ma_up_flag(df):
    # 同 index.html：3/5/10 日均線多頭天數 → 連續轉強的天數
    d3, d5, d10 = (pd.to_numeric(df[f'MA_UP_{n}d'], errors='coerce') for n in (3, 5, 10))
    return np.select([d3 == d10, (d3 == 3) & (d5 == 5), d3 == 3], [d10, d10, d5], 0)


def yahoo_url(code, market):
    suffix = 'TWO' if market == '上櫃' else 'TW'
    return f'https://tw.stock.yahoo.com/quote/{code}.{suffix}'


base = pd.read_csv(BASE_INFO, dtype=str).drop_duplicates('公司代號').set_index('公司代號')

if os.path.exists(LEDGER_PATH):
    ledger = pd.read_csv(LEDGER_PATH, dtype={'代號': str, '訊號日': str, '進場日': str,
                                             '最新日': str, '離場日': str})
    ledger[['名稱', '市場', '族群']] = ledger[['名稱', '市場', '族群']].fillna('')
else:
    ledger = pd.DataFrame(columns=LEDGER_COLS)

state = json.load(open(STATE_PATH, encoding='utf-8')) if os.path.exists(STATE_PATH) else {}
last_done = state.get('last_date', '')

daily_files = {os.path.basename(f)[:-5]: f for f in glob.glob(f'{DAILY_DIR}/*.xlsx')}
check_files = {os.path.basename(f)[:-5]: f for f in glob.glob(f'{CHECK_DIR}/*.xlsx')}
all_dates = sorted(daily_files)
# 第一次執行：用現有 DailyFiles 全部回補（不發 LINE，回補沒有副作用）
todo_dates = [d for d in all_dates if d > last_done]
print('上次處理到:', last_done or '(無)', '| 本次處理:', todo_dates)

In [ ]:
# =============================================================
# 3. 持倉更新：補進場價、檢查離場、記錄觀察期
# =============================================================
def _f(r, col):
    return pd.to_numeric(r.get(col), errors='coerce')


def update_positions(ledger, day, d, events):
    watch = (ledger['狀態'] == '持有中') | (pd.to_numeric(ledger['追蹤天數'], errors='coerce') < OBS_MAX)
    for i in ledger.index[watch]:
        code = ledger.at[i, '代號']
        if code not in day.index or d <= ledger.at[i, '訊號日']:
            continue                                   # 停牌/當天剛出訊號
        r = day.loc[code]
        close = float(r['收盤價'])

        # 訊號隔日開盤進場，確定停損價
        if pd.isna(ledger.at[i, '進場價']):
            entry = float(r['開盤價'])
            ledger.at[i, '進場日'] = d
            ledger.at[i, '進場價'] = entry
            ref = ledger.at[i, '量K低']
            floor = entry * (1 - MAX_LOSS)
            ledger.at[i, '停損價'] = round(max(ref, floor) if pd.notna(ref) else floor, 2)
            ledger.at[i, '持有天數'] = 0
            ledger.at[i, '追蹤天數'] = 0
            ledger.at[i, '最高收盤'] = close

        entry = float(ledger.at[i, '進場價'])
        ret = (close / entry - 1) * 100

        # 觀察期：進場後 OBS_MAX 日內，不論是否離場都記錄
        if int(ledger.at[i, '追蹤天數']) < OBS_MAX:
            n = int(ledger.at[i, '追蹤天數']) + 1
            ledger.at[i, '追蹤天數'] = n
            hi = (_f(r, '最高價') / entry - 1) * 100
            lo = (_f(r, '最低價') / entry - 1) * 100
            ledger.at[i, '期間最高%'] = round(np.fmax(ledger.at[i, '期間最高%'], hi), 2)
            ledger.at[i, '期間最低%'] = round(np.fmin(ledger.at[i, '期間最低%'], lo), 2)
            if n in OBS_DAYS:
                ledger.at[i, f'{n}日%'] = round(ret, 2)

        if ledger.at[i, '狀態'] != '持有中':
            continue

        hold = int(ledger.at[i, '持有天數']) + 1
        ledger.at[i, '持有天數'] = hold
        ledger.at[i, '最高收盤'] = max(float(ledger.at[i, '最高收盤']), close)
        ledger.at[i, '最新日'] = d
        ledger.at[i, '最新收盤'] = close
        ledger.at[i, '報酬%'] = round(ret, 2)

        peak = float(ledger.at[i, '最高收盤'])
        ma = _f(r, EXIT_MA)
        prev = _f(r, '前日收盤價')
        reason = ''
        if close < float(ledger.at[i, '停損價']):
            reason = '停損'
        elif (_f(r, '成交金額') >= _f(r, 'Volume_MA_5') * DUMP_VOL and close < float(r['開盤價'])
              and pd.notna(prev) and close <= prev * (1 - DUMP_DROP)):
            reason = '爆量黑K'
        elif pd.notna(ma) and close < ma:
            reason = '跌破' + EXIT_MA.replace('_', '')
        elif peak >= entry * (1 + TRAIL_TRIGGER) and close <= peak * (1 - TRAIL_DD):
            reason = '移動停利'
        elif hold >= MAX_HOLD_DAYS:
            reason = '時間出場'
        elif hold >= STALL_DAYS and ret < STALL_RET * 100:
            reason = '滯漲出場'

        if reason:
            ledger.at[i, '狀態'] = '已離場'
            ledger.at[i, '離場日'] = d
            ledger.at[i, '離場價'] = close
            ledger.at[i, '離場原因'] = reason
            events['exit'].append(i)
    return ledger

In [ ]:
# =============================================================
# 4. 新訊號：當日符合「有量」全部收進來
# =============================================================
def pick_new(ledger, ck, d, events):
    num = lambda c: pd.to_numeric(ck[c], errors='coerce')
    flag = pd.Series(ma_up_flag(ck), index=ck.index)
    ratio = num('成交金額') / num('Volume_MA_5')
    hit = (ratio > VOL_RATIO) & (num('_quote') > 0) & (num('LevelArea') > MIN_LEVEL) & (flag >= 1)
    holding = set(ledger.loc[ledger['狀態'] == '持有中', '代號'])
    rows = []
    for code, r in ck[hit].sort_values('成交金額', ascending=False).iterrows():
        if code in holding:
            continue                                   # 追蹤中不重複記錄
        close = float(r['收盤價'])
        low = _f(r, '最低價')
        info = base.loc[code] if code in base.index else {}
        score = _f(r, 'score')
        rows.append({
            '訊號日': d, '代號': code,
            '名稱': info.get('公司簡稱', '') if len(info) else str(r.get('stock_name', '')),
            '市場': info.get('Type', '') if len(info) else '',
            '族群': '' if pd.isna(r.get('族群')) else r.get('族群'),
            'score': round(score, 1) if pd.notna(score) else np.nan,
            '量比': round(ratio[code], 2), '漲幅%': round(_f(r, '_quote'), 2),
            'ma_up_flag': int(flag[code]),
            '訊號日收盤': close, '量K低': low,
            # 進場前先用訊號日收盤估停損，隔日開盤後再更新
            '停損價': round(max(low, close * (1 - MAX_LOSS)) if pd.notna(low) else close * (1 - MAX_LOSS), 2),
            '最新日': d, '最新收盤': close, '狀態': '持有中',
        })
    if rows:
        start = len(ledger)
        ledger = pd.concat([ledger, pd.DataFrame(rows)], ignore_index=True)
        events['new'].extend(range(start, len(ledger)))
    return ledger


events = {'new': [], 'exit': []}
for d in todo_dates:
    ledger = update_positions(ledger, load_day(daily_files[d]), d, events)
    if d in check_files:
        ledger = pick_new(ledger, load_day(check_files[d]), d, events)
    else:
        print(f'{d}：沒有 checkall，略過新訊號')

ledger = ledger.reindex(columns=LEDGER_COLS)
print('新訊號', len(events['new']), '| 離場', len(events['exit']),
      '| 追蹤中', int((ledger['狀態'] == '持有中').sum()))

In [ ]:
# =============================================================
# 5. 存檔 + 離場統計
# =============================================================
ledger.to_csv(LEDGER_PATH, index=False, encoding='utf-8-sig')
if todo_dates:
    state['last_date'] = todo_dates[-1]
    state['updated_at'] = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    json.dump(state, open(STATE_PATH, 'w', encoding='utf-8'), ensure_ascii=False, indent=1)

_closed = ledger[ledger['狀態'] == '已離場'].copy()
if len(_closed):
    _closed['_ret'] = (_closed['離場價'].astype(float) / _closed['進場價'].astype(float) - 1) * 100
    print(f"累計已離場 {len(_closed)} 筆｜勝率 {(_closed['_ret'] > 0).mean() * 100:.1f}%｜平均 {_closed['_ret'].mean():.2f}%")
    print(_closed.groupby('離場原因')['_ret'].agg(['count', 'mean']).round(2))
for n in OBS_DAYS:
    s = pd.to_numeric(ledger[f'{n}日%'], errors='coerce').dropna()
    if len(s):
        print(f'抱滿 {n} 日：{len(s)} 筆｜勝率 {(s > 0).mean() * 100:.1f}%｜平均 {s.mean():.2f}%')
ledger.tail(10)

In [ ]:
# =============================================================
# 5b. 網頁資料：依「訊號月份」拆檔，給 volume.html 讀取
#     Result/volume/web/index.json      月份清單 + 離場規則
#     Result/volume/web/YYYY-MM.json    該月訊號的所有紀錄
# =============================================================
WEB_DIR = f'{OUT_DIR}/web'
os.makedirs(WEB_DIR, exist_ok=True)


def _chart_path(code):
    # 對應 _genHtml_cyber 的 save_plt_to_html 檔名：Html/[代號]簡稱-產業-上市/上櫃.html
    if code not in base.index:
        return ''
    r = base.loc[code]
    ind = r.get('上櫃公司產業類別' if r.get('Type') == '上櫃' else '上市公司產業類別', '')
    return f"Html/[{code}]{r.get('公司簡稱', '')}-{ind}-{r.get('Type', '')}.html".replace('*', '')


def _num(v, nd=2):
    return None if pd.isna(v) else round(float(v), nd)


def _str(v):
    return '' if pd.isna(v) else v


web_rows = []
for _, r in ledger.iterrows():
    web_rows.append({
        '訊號日': r['訊號日'], '代號': r['代號'], '名稱': r['名稱'], '族群': _str(r['族群']),
        'score': _num(r['score'], 1), '量比': _num(r['量比']), '漲幅%': _num(r['漲幅%']),
        '收盤': _num(r['訊號日收盤']), '停損': _num(r['停損價']), '進場價': _num(r['進場價']),
        '報酬%': _num(r['報酬%']),
        '持有天數': None if pd.isna(r['持有天數']) else int(r['持有天數']),
        '狀態': '追蹤中' if r['狀態'] == '持有中' else '離場',
        '離場日': _str(r['離場日']), '離場原因': _str(r['離場原因']),
        '最高%': _num(r['期間最高%']), '最低%': _num(r['期間最低%']),
        **{f'{n}日%': _num(r[f'{n}日%']) for n in OBS_DAYS},
        'yahoo': yahoo_url(r['代號'], r['市場']), 'chart': _chart_path(r['代號']),
    })

months = {}
for row in web_rows:
    months.setdefault(row['訊號日'][:7], []).append(row)
for m, rows in months.items():
    rows.sort(key=lambda x: (x['訊號日'], x['代號']), reverse=True)
    json.dump(rows, open(f'{WEB_DIR}/{m}.json', 'w', encoding='utf-8'), ensure_ascii=False, separators=(',', ':'))

index = {
    'updated_at': datetime.now().strftime('%Y-%m-%d %H:%M'),
    'last_date': state.get('last_date', ''),
    'obs_days': list(OBS_DAYS),
    'rules': [
        f'停損：收盤 < 有量K低（最多 -{MAX_LOSS:.0%}）',
        f'爆量黑K：量 ≥ 5日均×{DUMP_VOL:g}、收黑跌 ≥ {DUMP_DROP:.0%}',
        f'跌破 {EXIT_MA.replace("_", "")}',
        f'移動停利：獲利 ≥ {TRAIL_TRIGGER:.0%} 後回落 {TRAIL_DD:.0%}',
        f'時間出場：持有 {MAX_HOLD_DAYS} 日',
        f'滯漲：持有 {STALL_DAYS} 日報酬 < {STALL_RET:.0%}',
    ],
    'months': [{'month': m, 'count': len(rows),
                'active': sum(x['狀態'] == '追蹤中' for x in rows)}
               for m, rows in sorted(months.items(), reverse=True)],
}
json.dump(index, open(f'{WEB_DIR}/index.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=1)
print('網頁資料：', {m['month']: m['count'] for m in index['months']})